In [3]:
from google.colab import drive, auth
drive.mount("/content/drive")

auth.authenticate_user()

from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import os

drive_api = build("drive", "v3")

MANIFEST_PATH = "/content/drive/MyDrive/image_manifest.csv"
OLD_MAPPING_PATH = (
    "/content/drive/MyDrive/image_similarity_artifacts/"
    "embedding_chunks/fashion_mapping.npz"
)

assert os.path.exists(MANIFEST_PATH), f"Missing: {MANIFEST_PATH}"
assert os.path.exists(OLD_MAPPING_PATH), f"Missing: {OLD_MAPPING_PATH}"

manifest = pd.read_csv(MANIFEST_PATH)
old_mapping = np.load(OLD_MAPPING_PATH)

print("Manifest columns:", manifest.columns.tolist())
print("Manifest rows:", len(manifest))
print("Mapping keys:", old_mapping.files)
print("Mapping image IDs:", old_mapping["image_ids"].shape)
print("Sample paths:", manifest["drive_path"].head().tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifest columns: ['image_id', 'category', 'drive_path']
Manifest rows: 46994
Mapping keys: ['image_ids', 'image_paths']
Mapping image IDs: (46994,)
Sample paths: ['/content/drive/MyDrive/mydata/Dresses/10016.jpg', '/content/drive/MyDrive/mydata/Dresses/10360.jpg', '/content/drive/MyDrive/mydata/Dresses/10365.jpg', '/content/drive/MyDrive/mydata/Dresses/10406.jpg', '/content/drive/MyDrive/mydata/Dresses/10500.jpg']


In [4]:
response = drive_api.files().list(
    q="name = 'mydata' and mimeType = 'application/vnd.google-apps.folder' "
      "and trashed = false",
    fields="files(id, name, parents)",
    pageSize=100
).execute()

folders = response.get("files", [])

for folder in folders:
    print(folder)

{'parents': ['0ALIHJr1oiCsCUk9PVA'], 'id': '1W6OhvRKrFYXHFENtOoZzullgSYLxzKfy', 'name': 'mydata'}


In [7]:
ROOT_FOLDER_ID = "1W6OhvRKrFYXHFENtOoZzullgSYLxzKfy"

try:
    folder = drive_api.files().get(
        fileId=ROOT_FOLDER_ID,
        fields="id,name,mimeType,parents"
    ).execute()

    print(folder)

except Exception as e:
    print(type(e).__name__)
    print(e)

{'id': '1W6OhvRKrFYXHFENtOoZzullgSYLxzKfy', 'name': 'mydata', 'mimeType': 'application/vnd.google-apps.folder', 'parents': ['0ALIHJr1oiCsCUk9PVA']}


In [9]:
response = drive_api.files().list(
    q=" '1W6OhvRKrFYXHFENtOoZzullgSYLxzKfy' in parents and trashed = false",
    fields="nextPageToken,files(id, name, mimeType)",
    pageSize=100
).execute()

items = response.get("files", [])

print("Items found:", len(items))

for item in items:
    print(item["name"], "|", item["mimeType"], "|", item["id"])

Items found: 4
women fashion | application/vnd.google-apps.folder | 1ByPBqW6yD-_N5HKn9HP-iUP4od90IKKu
mixed | application/vnd.google-apps.folder | 1AnEOOXooQNzAtBP1FDzBQN2wrRlvW6YO
Dresses | application/vnd.google-apps.folder | 1yBORUmWdBa6Xlis1ep6EWxaCHGfTaiLw
tshirt | application/vnd.google-apps.folder | 1pKY3N_TiVb08AaEWLPSEnh5A18-UkUm1


In [10]:
def list_drive_images(folder_id):
    files_by_path = {}
    folders_to_visit = [(folder_id, "")]

    while folders_to_visit:
        current_id, relative_dir = folders_to_visit.pop()
        page_token = None

        while True:
            response = drive_api.files().list(
                q=f"'{current_id}' in parents and trashed = false",
                fields="nextPageToken,files(id,name,mimeType)",
                pageSize=1000,
                pageToken=page_token
            ).execute()

            for item in response.get("files", []):
                name = item["name"]
                relative_path = f"{relative_dir}/{name}".lstrip("/")

                if item["mimeType"] == "application/vnd.google-apps.folder":
                    folders_to_visit.append((item["id"], relative_path))
                elif item["mimeType"].startswith("image/"):
                    files_by_path[relative_path] = item["id"]

            page_token = response.get("nextPageToken")
            if not page_token:
                break

    return files_by_path


drive_files = list_drive_images(ROOT_FOLDER_ID)

print("Total images found:", len(drive_files))
print("\nFirst 10 image paths:")

for path, file_id in list(drive_files.items())[:10]:
    print(path, "->", file_id)

Total images found: 46994

First 10 image paths:
tshirt/977.jpg -> 1lcdiryLT804hpMPBaaCqU_y-mqWQ1M8T
tshirt/919.jpg -> 1N4KUVXrmA7hrqGW8lCvi-wQewajrzc7r
tshirt/979.jpg -> 1E7CPCQhB9oIJ3YqVGvHcbzyAChlwToJ6
tshirt/933.jpg -> 1JAXDMDq-ZP76YMtLo_UNlIwEtQBZ8wPD
tshirt/972.jpg -> 1PklFWCgITJV98TQwbe80DX0MAYcMrb5Z
tshirt/98.jpg -> 11BhwQllmEU-rZhbleGjyJYrmJYmOob-G
tshirt/964.jpg -> 1cu8Kq6ZW0EcQS21mZlVKD4Z0AGzyD2UR
tshirt/981.jpg -> 1c21yfQxgqIG6NFDPNE72a6aMlJCUX5tn
tshirt/927.jpg -> 16cTtfG8cnzJY7HoIuIoR8gqipfnZGeCV
tshirt/922.jpg -> 11Qn32HJeHpfBK4ISJ2fTcVzK5bpdut7R


In [11]:
import pandas as pd
import os

MANIFEST_PATH = "/content/drive/MyDrive/image_manifest.csv"
OLD_MAPPING_PATH = (
    "/content/drive/MyDrive/image_similarity_artifacts/"
    "embedding_chunks/fashion_mapping.npz"
)

manifest = pd.read_csv(MANIFEST_PATH)
old_mapping = np.load(OLD_MAPPING_PATH)

print("Manifest rows:", len(manifest))
print("Manifest columns:", manifest.columns.tolist())
print("\nManifest sample:")
print(manifest[["image_id", "drive_path"]].head(10).to_string(index=False))

print("\nExisting mapping keys:", old_mapping.files)
print("Existing image IDs:", old_mapping["image_ids"].shape)

Manifest rows: 46994
Manifest columns: ['image_id', 'category', 'drive_path']

Manifest sample:
 image_id                                      drive_path
        0 /content/drive/MyDrive/mydata/Dresses/10016.jpg
        1 /content/drive/MyDrive/mydata/Dresses/10360.jpg
        2 /content/drive/MyDrive/mydata/Dresses/10365.jpg
        3 /content/drive/MyDrive/mydata/Dresses/10406.jpg
        4 /content/drive/MyDrive/mydata/Dresses/10500.jpg
        5 /content/drive/MyDrive/mydata/Dresses/11261.jpg
        6 /content/drive/MyDrive/mydata/Dresses/12830.jpg
        7 /content/drive/MyDrive/mydata/Dresses/12835.jpg
        8 /content/drive/MyDrive/mydata/Dresses/12837.jpg
        9 /content/drive/MyDrive/mydata/Dresses/12849.jpg

Existing mapping keys: ['image_ids', 'image_paths']
Existing image IDs: (46994,)


In [12]:
import posixpath
import numpy as np

def normalize_path(path):
    prefix = "/content/drive/MyDrive/mydata/"
    path = str(path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return posixpath.normpath(path).lstrip("/")


manifest["relative_path"] = manifest["drive_path"].map(normalize_path)

# Match each manifest path to its Google Drive file ID.
manifest["drive_file_id"] = manifest["relative_path"].map(drive_files)

matched = manifest["drive_file_id"].notna().sum()
missing = manifest["drive_file_id"].isna().sum()

print("Manifest images:", len(manifest))
print("Matched images:", matched)
print("Missing images:", missing)

if missing:
    print("\nUnmatched paths:")
    print(
        manifest.loc[
            manifest["drive_file_id"].isna(),
            ["image_id", "relative_path"]
        ].head(10).to_string(index=False)
    )
else:
    print("\nAll images matched successfully!")

Manifest images: 46994
Matched images: 46994
Missing images: 0

All images matched successfully!


In [13]:
image_ids = old_mapping["image_ids"]

if len(np.unique(image_ids)) != len(image_ids):
    raise ValueError("Duplicate image IDs found")

manifest_by_id = manifest.set_index("image_id")
ordered = manifest_by_id.reindex(image_ids)

if ordered["drive_file_id"].isna().any():
    raise ValueError("Missing Drive IDs after aligning image IDs")

OUTPUT_PATH = (
    "/content/drive/MyDrive/image_similarity_artifacts/"
    "embedding_chunks/fashion_mapping_drive.npz"
)

np.savez_compressed(
    OUTPUT_PATH,
    image_ids=image_ids,
    drive_file_ids=ordered["drive_file_id"].to_numpy(dtype=str),
    image_paths=ordered["relative_path"].to_numpy(dtype=str),
)

print("Saved:", OUTPUT_PATH)
print("Total mappings:", len(image_ids))
print("\nFirst five mappings:")

for image_id, path, file_id in zip(
    image_ids[:5],
    ordered["relative_path"].to_numpy()[:5],
    ordered["drive_file_id"].to_numpy()[:5]
):
    print(f"ID={image_id} | {path} | {file_id}")

Saved: /content/drive/MyDrive/image_similarity_artifacts/embedding_chunks/fashion_mapping_drive.npz
Total mappings: 46994

First five mappings:
ID=0 | Dresses/10016.jpg | 14TwkCjZii_B0f9Lp2sFxbInGTEiXmYT9
ID=1 | Dresses/10360.jpg | 150oTFs8q_HcRtrJ2_toeXBKPDWLqHs3_
ID=2 | Dresses/10365.jpg | 1SwY-2863BbEdAiigd0QpD8dFZuvyy_S1
ID=3 | Dresses/10406.jpg | 1qWcksr1zH9MF5o6dxgUN9AH_UmOq_p76
ID=4 | Dresses/10500.jpg | 1tPZ7_NZEw3t5T88FyGMTTt1G6_XNTCv5


In [14]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/image_similarity_artifacts/"
    "embedding_chunks/fashion_mapping_drive.npz"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>